# Sunday Project | Data Quality, Insights & CSV Operations
**Student:** Safwan  
**Training ID:** 307

This notebook completes both assignment parts using `sales_raw.csv`. Run cells top-to-bottom in Jupyter or Google Colab.

## Part A — Data Quality & Basic Insights (70 marks)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
raw = pd.read_csv('sales_raw.csv')
print('First five rows:'); display(raw.head())
print('Shape:', raw.shape)
print('Columns:', raw.columns.tolist())
print('Data types:
', raw.dtypes)
print('Missing values per column:
', raw.isna().sum())
print('Duplicate complete rows:', raw.duplicated().sum())
print('Duplicate transaction IDs:', raw.duplicated(subset='transaction_id').sum())

### Cleaning decisions
Trim and standardize text using title case; label missing region as Unknown. Parse dates and numeric columns, coercing invalid values. Fill missing/invalid quantity and unit price with the median for the product category (overall median fallback). Remove duplicate transaction IDs keeping the first, invalid-date rows, and non-positive quantity rows. Retain and flag the unusually high $10,000 price for review rather than assuming it is erroneous.

In [ ]:
df = raw.copy()
df['category'] = df['category'].astype('string').str.strip().str.title()
df['region'] = df['region'].astype('string').str.strip().str.title().replace({'': pd.NA, '<NA>': pd.NA}).fillna('Unknown')
df['sale_date'] = pd.to_datetime(df['sale_date'], format='%m/%d/%Y', errors='coerce')
df['quantity'] = pd.to_numeric(df['quantity'], errors='coerce')
df['unit_price'] = pd.to_numeric(df['unit_price'], errors='coerce')
rows_before = len(df)
df = df.drop_duplicates(subset='transaction_id', keep='first').copy()
for col in ['quantity', 'unit_price']:
    category_median = df.groupby('category', dropna=False)[col].transform('median')
    df[col] = df[col].fillna(category_median).fillna(df[col].median())
df = df.dropna(subset=['sale_date'])
df = df[df['quantity'] > 0].copy()
df['quantity'] = df['quantity'].astype(int)
df['revenue'] = df['quantity'] * df['unit_price']
q1, q3 = df['unit_price'].quantile([0.25, 0.75]); outlier_limit = q3 + 1.5 * (q3 - q1)
df['outlier_price_flag'] = df['unit_price'] > outlier_limit
df = df.sort_values(['sale_date', 'transaction_id']).reset_index(drop=True)
print(f'Rows before cleaning: {rows_before}; rows after cleaning: {len(df)}')
print('Cleaned missing values:
', df.isna().sum()); display(df.head())

In [ ]:
revenue_by_region = df.groupby('region', as_index=False).agg(transactions=('transaction_id','count'), units_sold=('quantity','sum'), revenue=('revenue','sum')).sort_values('revenue', ascending=False)
revenue_by_category = df.groupby('category', as_index=False).agg(transactions=('transaction_id','count'), units_sold=('quantity','sum'), revenue=('revenue','sum')).sort_values('revenue', ascending=False)
print('Revenue by region:'); display(revenue_by_region)
print('Revenue by category:'); display(revenue_by_category)
display(df[['quantity','unit_price','revenue']].describe())

In [ ]:
ax = revenue_by_region.sort_values('revenue').plot.barh(x='region', y='revenue', legend=False, color='#276678', figsize=(8,5))
ax.set(title='Revenue by Region', xlabel='Revenue', ylabel='Region'); plt.tight_layout(); plt.show()
ax = revenue_by_category.sort_values('revenue').plot.barh(x='category', y='revenue', legend=False, color='#e07a5f', figsize=(8,5))
ax.set(title='Revenue by Product Category', xlabel='Revenue', ylabel='Category'); plt.tight_layout(); plt.show()

### Evidence-based findings
1. North generated the highest regional revenue ($11,470), above South ($1,145). North’s result is strongly affected by transaction 1016, whose $10,000 stationery price was retained but flagged for verification.
2. Stationery generated the highest category revenue ($10,570), compared with Food ($2,435) and Books ($1,770). This result is also driven by the flagged transaction, so interpret it with the outlier caveat.

In [ ]:
df.to_csv('cleaned_sales.csv', index=False)
print('Saved cleaned_sales.csv')

## Part B — CSV File Handling & Manipulation (30 marks)

In [ ]:
sales = pd.read_csv('cleaned_sales.csv')
sales['tax_amount'] = (sales['revenue'] * 0.05).round(2)
print('First five rows after adding tax:'); display(sales.head())
books_mask = sales['category'].eq('Books')
sales.loc[books_mask, 'unit_price'] = sales.loc[books_mask, 'unit_price'] * 1.10
sales['unit_price'] = sales['unit_price'].round(2)\nsales['revenue'] = (sales['quantity'] * sales['unit_price']).round(2)
sales['tax_amount'] = (sales['revenue'] * 0.05).round(2)
selected_transaction_id = sales['transaction_id'].iloc[0]
sales = sales.loc[sales['transaction_id'] != selected_transaction_id].copy()
print('Removed transaction_id:', selected_transaction_id)
sales['review_status'] = 'Pending review'
sales = sales.drop(columns=['review_status'])
sales.to_csv('sales_updated.csv', index=False)
verified = pd.read_csv('sales_updated.csv')
print('First five saved rows:'); display(verified.head())
print('Final column names:', verified.columns.tolist())